# Debug: #763 Stage 2 — wire `engine='vector'` into Lattice

Stage 1 (`find_lines_from_layout`) merged in #764 — it reads ruled lines straight from playa's layout tree. **Stage 2** wires it into the Lattice parser behind an `engine=` kwarg so we skip rasterise→OpenCV for PDFs with native vector lines.

This notebook is the scratchpad for that work: it (a) shows `find_lines_from_layout` output next to the raster `find_lines` output on the same PDF, (b) benchmarks the two, (c) sketches the `_generate_table_bbox` vector path.

Issue: [#763](https://github.com/camelot-dev/camelot/issues/763) · Stage 1 PR: #764 · **Setup:** `pip install -e .[plot,base]`.

In [ ]:
import camelot, os
import playa
from camelot.image_processing import find_lines_from_layout
from camelot.utils import get_page_layout

PDF = 'tests/files/foo.pdf'   # has a ruled 7x7 grid
print('camelot', camelot.__version__)

## Step 1 — vector lines vs raster lines on the same page

Pull horizontal + vertical lines via the Stage-1 helper (PDF coords) and via the raster `find_lines` (image coords). They should describe the same grid.

In [ ]:
with playa.open(PDF, space='page') as pdf:
    page = pdf.pages[0]
    layout, dims = get_page_layout(page)
    vh = find_lines_from_layout(layout, direction='horizontal')
    vv = find_lines_from_layout(layout, direction='vertical')
print(f'vector: {len(vh)} horizontal, {len(vv)} vertical lines (PDF coords)')
for ln in sorted(vh)[:10]:
    print('  H', tuple(round(c, 1) for c in ln))
for ln in sorted(vv)[:10]:
    print('  V', tuple(round(c, 1) for c in ln))

## Step 2 — timing: vector extraction vs full raster pipeline

The whole point of Stage 2. Time `find_lines_from_layout` (no rasterise) against the current lattice `_generate_table_bbox` (rasterise + threshold + cv2.findContours).

In [ ]:
import time

# Vector path timing (layout already parsed once).
with playa.open(PDF, space='page') as pdf:
    page = pdf.pages[0]
    layout, dims = get_page_layout(page)
    t0 = time.perf_counter()
    for _ in range(20):
        find_lines_from_layout(layout, direction='horizontal')
        find_lines_from_layout(layout, direction='vertical')
    t_vector = (time.perf_counter() - t0) / 20
print(f'vector line-extraction: {t_vector*1000:.2f} ms/page')

# Raster path timing (full read_pdf lattice).
t0 = time.perf_counter()
for _ in range(3):
    camelot.read_pdf(PDF, flavor='lattice')
t_raster = (time.perf_counter() - t0) / 3
print(f'full lattice read_pdf: {t_raster*1000:.2f} ms/page (includes parse + table build)')

## Step 3 — sketch the engine='vector' wiring

The integration plan for `camelot/parsers/lattice.py`:

1. Add `engine='raster'` (default) to `Lattice.__init__`; store `self.engine`.
2. In `_generate_table_bbox`, branch:
   - `engine='raster'` → current path (unchanged).
   - `engine='vector'` → `find_lines_from_layout(self.layout, ...)`, convert PDF→image coords with the existing `scale_pdf`, then either synthesise a thin mask for `find_contours` OR write `find_contours_from_lines` (compute intersections directly).
   - `engine='auto'` → probe `self.layout` for any LTLine/stroked-LTRect; if present use vector, else raster.
3. Add `engine` to `lattice_kwargs` in `utils.py` so `validate_input` accepts it.
4. Integration test: assert `read_pdf(foo.pdf, flavor='lattice', engine='vector')` matches `engine='raster'` output on the lattice fixtures that have native vector lines.

Prototype `find_contours_from_lines` below.

In [ ]:
# Prototype: table bbox from line intersections (vector-native, no cv2).
def find_contours_from_lines(h_lines, v_lines, join_tol=2.0):
    """Group h/v lines into rectangular table regions via their span.
    Returns (x, y, w, h) tuples mirroring image_processing.find_contours.
    This is a sketch — refine the grouping for multi-table pages."""
    if not h_lines or not v_lines:
        return []
    xs = [x for ln in v_lines for x in (ln[0], ln[2])]
    ys = [y for ln in h_lines for y in (ln[1], ln[3])]
    x0, x1 = min(xs), max(xs)
    y0, y1 = min(ys), max(ys)
    return [(x0, y0, x1 - x0, y1 - y0)]

print(find_contours_from_lines(vh, vv))